# State Management

## What's covered

- **Why state exists** — the mapping from resource addresses to real cloud objects
- **What's inside** `terraform.tfstate` and why it's sensitive
- **Local vs remote backends** — local is fine until it's not, and "not" comes fast in teams
- The standard **S3 + DynamoDB** remote-backend pattern, in full
- **Terraform Cloud / HCP** as the managed alternative
- **State locking** — how the lock works and what to do when it's stuck
- **Drift** — when the cloud and the state file disagree
- **Sensitive values** and how they leak through state
- **State surgery** — `terraform state list / show / mv / rm` and `terraform import` / the `import` block
- The **bootstrapping** problem and the standard solution


## Why state exists

Terraform is declarative — your `.tf` files describe *what should exist*. But Terraform also has to know *what currently exists*, so it can compute the diff.

The trivial-but-wrong answer is "ask the cloud every time." That works for a single resource type but breaks at scale: how does Terraform know which of the 10,000 EC2 instances in your account it's responsible for managing? Some belong to you; others belong to a different team; others to a side project that doesn't use Terraform at all.

The **state file** is Terraform's answer. It records, for every resource address in your config, the corresponding *real cloud object*: its ID, its computed attributes, its dependencies. State maps `aws_s3_bucket.logs` to "the bucket whose ID is `tf-hello-a3f9c1b2`, whose ARN is `arn:aws:s3:::tf-hello-a3f9c1b2`, whose region is us-east-1, ..."

```
   .tf file                                  state file
   --------                                  ----------

   resource "aws_s3_bucket" "logs" {  <--->  {
     bucket = "my-app-logs"                    "address": "aws_s3_bucket.logs",
   }                                           "instances": [{
                                                 "attributes": {
                                                   "id":     "my-app-logs",
                                                   "arn":    "arn:aws:s3:::my-app-logs",
                                                   "region": "us-east-1",
                                                   ...
                                                 }
                                               }]
                                             }
```

Three things state lets Terraform do:

- **Identify resources to manage.** Anything in state is managed by this configuration. Anything not in state isn't.
- **Skip the refresh round-trip per resource.** State holds the last-known attributes; refresh updates them but starts from a baseline.
- **Detect drift.** Cloud says one thing; state says another; the diff is drift.

Without state, every plan would have to scan the entire cloud account and reconstruct ownership — slow, expensive, and ambiguous.


## What's actually in the state file

`terraform.tfstate` is JSON. A toy example:

```json
{
  "version": 4,
  "terraform_version": "1.6.5",
  "serial": 12,
  "lineage": "f8c5d8e2-...",
  "outputs": {
    "bucket_name": {
      "value": "tf-hello-a3f9c1b2",
      "type": "string"
    }
  },
  "resources": [
    {
      "mode": "managed",
      "type": "aws_s3_bucket",
      "name": "hello",
      "provider": "provider[\"registry.terraform.io/hashicorp/aws\"]",
      "instances": [
        {
          "schema_version": 0,
          "attributes": {
            "id":     "tf-hello-a3f9c1b2",
            "arn":    "arn:aws:s3:::tf-hello-a3f9c1b2",
            "bucket": "tf-hello-a3f9c1b2",
            "region": "us-east-1",
            "tags":   {}
          }
        }
      ]
    }
  ]
}
```

Five fields worth knowing.

- **`serial`** — a monotonically increasing counter, incremented on every write. Used to detect concurrent modification.
- **`lineage`** — a UUID identifying *this* state file. Cloning a state file gives the same lineage; making a new one (via `init` against a fresh backend) gives a different lineage. Mismatched lineage between local and remote is one of the louder errors.
- **`outputs`** — the values exported by `output` blocks, cached so other states can `terraform_remote_state` them.
- **`resources`** — the heart of the file. One entry per resource address, with one or more *instances* (multiple if `count` or `for_each` is used). Each instance carries every attribute the provider returned.
- **`schema_version`** — the provider's idea of what shape the attributes are in. Provider upgrades occasionally migrate this number forward.

**Critical:** state can contain **secrets**. Database passwords, IAM access keys, TLS certificates — anything you write into a resource that the provider stores in attributes ends up in plain text in the state file. That's the single biggest reason the file needs to be stored carefully and never committed to git.


## Local state — fine until it's not

When you run `terraform init` without configuring a backend, Terraform uses the **local** backend: state is written to `terraform.tfstate` in the working directory.

This works for one person experimenting. It breaks the moment a second person needs to apply against the same infrastructure:

- **No shared state.** Alice's `terraform.tfstate` and Bob's are independent files. Alice creates 3 resources; her state shows them. Bob's state still shows zero. When Bob runs apply, he tries to recreate everything — and either it errors out (name conflict) or the cloud now has duplicate resources Alice's state doesn't know about.
- **No locking.** Two `apply` runs concurrently *will* corrupt state. The two write the file at different points; one wins, the other's changes are lost.
- **Easy to lose.** Laptop dies, working directory deleted, accident in `rm`. The state file is gone, and Terraform has no record of what it created. The cloud resources still exist; you've just lost your map to them.

The solution is a **remote backend**: state lives in a shared store (S3, Terraform Cloud, Azure Storage, GCS, Consul) and is locked during writes.

**The honest rule.** Use a remote backend from day one of any team work, and from the moment you create production infrastructure. The cost is 20 minutes of setup; the savings are sleepless nights.


## The S3 + DynamoDB pattern

The canonical AWS remote-backend setup. State in S3 (durable, versioned), locking in DynamoDB (consistent, cheap):

```hcl
terraform {
  backend "s3" {
    bucket         = "myorg-terraform-state"
    key            = "platform/network/terraform.tfstate"
    region         = "us-east-1"
    dynamodb_table = "myorg-terraform-locks"
    encrypt        = true
  }
}
```

Walk through the arguments:

- **`bucket`** — the S3 bucket that holds state. One bucket per organization is typical; the `key` field distinguishes states within it.
- **`key`** — the path to *this* config's state file inside the bucket. Convention: organize by `<system>/<component>/terraform.tfstate`. The pattern matters because notebook 06 covers splitting state by environment using this key.
- **`region`** — the bucket's region. Doesn't have to match the resources you're managing.
- **`dynamodb_table`** — the DynamoDB table holding lock records. One per organization is fine.
- **`encrypt = true`** — encrypts state at rest in S3 (SSE-S3 by default; specify `kms_key_id` for KMS).

The **S3 bucket must have versioning enabled** so you can roll back a corrupted state. The **DynamoDB table** must have a primary key named exactly `LockID` (string).

You only set this up once per organization. The bucket and table that *hold* state for everything else are usually created by a tiny **bootstrap configuration** with local state (then you migrate). We come back to this at the end of the notebook.


## Terraform Cloud (HCP Terraform) — the managed alternative

HashiCorp's hosted backend, now branded **HCP Terraform**. State, locking, runs, RBAC, audit log, secret variables — all managed.

```hcl
terraform {
  cloud {
    organization = "myorg"
    workspaces {
      name = "platform-network-prod"
    }
  }
}
```

What you get:

- **State management** — versioned, encrypted, no S3 bucket to babysit.
- **Locking** — built-in, no DynamoDB table.
- **Remote runs** — apply runs on HCP's workers, not your laptop. Centralized logs.
- **Variable storage** — sensitive variables stored encrypted, injected into runs.
- **RBAC and SSO** — gate plan and apply by team membership.
- **Policy enforcement (paid tier)** — Sentinel or OPA policies that gate applies.

The cost: vendor lock-in (you can leave, but it's another migration) and the BSL license consideration since Terraform 1.6. OpenTofu users typically use S3 + DynamoDB or Scalr / Spacelift / Env0 as alternatives.

For new teams without strong opinions, HCP Terraform is the fastest path to a production-safe setup. For teams already on AWS with strong S3 instincts, S3 + DynamoDB is just fine.


## State locking

When Terraform writes state, it first acquires a **lock** in the backend's locking mechanism. Two `apply` runs cannot proceed concurrently — the second one blocks (or fails) until the first releases.

For the S3 + DynamoDB backend, locking happens via a write to the DynamoDB table:

```
DynamoDB item during a lock:

  LockID  : "myorg-terraform-state/platform/network/terraform.tfstate"
  Info    : {
    "ID":        "abc123-def456",
    "Operation": "OperationTypeApply",
    "Who":       "alice@laptop",
    "Created":   "2025-06-09T14:32:01Z",
    "Path":      "platform/network/terraform.tfstate"
  }
```

When apply completes, Terraform deletes the item. When the next run tries to acquire the lock, it sees the item gone and proceeds.

**What can go wrong:**

- **Crashed apply.** The process dies (Ctrl-C twice, network drop, OOM kill) before releasing. The lock item lingers. The next run errors out with "state is locked" plus the lock holder's info.
- **The fix:** `terraform force-unlock <LOCK_ID>`. The lock ID is in the error message. Run it only after *confirming no one is actually applying* — force-unlocking a live run corrupts state.
- **Backend misconfiguration** — DynamoDB table doesn't exist, IAM permission denied. Init fails before lock is even attempted. Fix the backend setup.

Some shops set a **lock timeout** with `terraform apply -lock-timeout=5m` so CI runs wait politely instead of immediately failing. The default timeout is zero (immediate failure).


## Drift — when reality disagrees

Drift is when the actual cloud state differs from the state file. Causes:

- Someone changed something in the console.
- An external tool modified a tagged attribute.
- A failed `apply` left a half-modified resource.
- A provider bug caused a silent attribute drift.

**Detecting drift:**

```bash
$ terraform plan -refresh-only
```

This refreshes every resource from the cloud (updating the state file's view) and reports any differences from the previous state. No changes to config are considered; this is *only* drift. The output is the diff between state-before-refresh and state-after-refresh.

```
$ terraform plan -refresh-only

# aws_instance.web has been changed
~ resource "aws_instance" "web" {
    id            = "i-0abc123"
  ~ instance_type = "t3.micro" -> "t3.small"
    tags          = { "Name" = "web" }
  ...
}
```

Someone resized the instance in the console. Terraform now knows; you decide whether to update the HCL to match (accept the drift) or run `terraform apply` to revert it.

**Production-grade drift detection** is a scheduled CI job that runs `terraform plan -refresh-only -detailed-exitcode` periodically (nightly, hourly). Exit code 2 means "drift detected"; the job pages an oncall. AWS Config, CloudTrail-based custom rules, or third-party tools (env0, Spacelift, Cloudquery) all hook into this idea.


## Sensitive values — how state leaks them

Terraform tries to mask sensitive values in *plan output* using the `sensitive = true` argument on variables and outputs (and certain provider-marked attributes). But the values are *still in the state file*, in plain text.

```hcl
variable "db_password" {
  type      = string
  sensitive = true
}

resource "aws_db_instance" "main" {
  password = var.db_password
  # ...
}
```

The plan output shows `password = (sensitive value)`. The state file shows `"password": "the-actual-password-string"`.

**Three rules for handling secrets:**

- **Never commit state to git.** Even if your `.gitignore` is correct, never trust it for a state file with secrets. Use a remote backend with encryption.
- **Encrypt state at rest.** `encrypt = true` on the S3 backend; HCP Terraform encrypts by default.
- **Limit who can read state.** S3 bucket policy, KMS key access, HCP team permissions. State-read access *is* secret-read access.

For the secrets themselves: pull from a secrets manager rather than passing through Terraform variables.

```hcl
data "aws_secretsmanager_secret_version" "db_password" {
  secret_id = "myorg/db/password"
}

resource "aws_db_instance" "main" {
  password = data.aws_secretsmanager_secret_version.db_password.secret_string
}
```

The password is still in state, but at least the *source of truth* is the secrets manager — rotation happens there and Terraform picks up the new value next apply. This is the modern best practice.


## State surgery — the commands

When the configuration and the state disagree in a way `apply` can't fix on its own, you reach for the `terraform state` subcommands. **Always back up state before running any of these.**

### `terraform state list`

Print every resource address in state.

```bash
$ terraform state list
aws_internet_gateway.main
aws_route_table.public
aws_route_table_association.public["public_a"]
aws_route_table_association.public["public_b"]
aws_route_table_association.public["public_c"]
aws_subnet.public["public_a"]
aws_subnet.public["public_b"]
aws_subnet.public["public_c"]
aws_vpc.main
```

The starting point for any state-surgery task — "what's actually in there?"

### `terraform state show <address>`

Print one resource's attributes from state.

```bash
$ terraform state show aws_vpc.main
# aws_vpc.main:
resource "aws_vpc" "main" {
    arn        = "arn:aws:ec2:us-east-1:123456789012:vpc/vpc-0abc123"
    cidr_block = "10.0.0.0/16"
    id         = "vpc-0abc123"
    ...
}
```

Useful for verifying what Terraform *thinks* exists, especially before a destructive operation.

### `terraform state pull` / `push`

`pull` writes the entire state file to stdout. `push` uploads a local file as the new state. The duo lets you edit state by hand:

```bash
$ terraform state pull > state.json
$ vim state.json   # edit carefully
$ terraform state push state.json
```

Use only when the higher-level commands (`mv`, `rm`) can't do what you need. Direct edits are how state gets corrupted.


### `terraform state mv` — renaming

When you rename a resource in HCL (`aws_s3_bucket.logs` → `aws_s3_bucket.access_logs`), Terraform doesn't know it's the same resource. The plan shows the old one being destroyed and a new one created — disaster for stateful resources.

`terraform state mv` tells Terraform "the resource at address A is the same thing as address B; update state without touching the cloud."

```bash
$ terraform state mv aws_s3_bucket.logs aws_s3_bucket.access_logs
```

After this, `terraform plan` shows no changes — state and config agree on the new address.

**Modern alternative (Terraform 1.1+):** the `moved` block, declared inline:

```hcl
moved {
  from = aws_s3_bucket.logs
  to   = aws_s3_bucket.access_logs
}
```

This survives a fresh checkout — it's part of the configuration. State surgery via `state mv` requires the operator to remember to run the command; `moved` blocks are self-documenting and reproducible. **Prefer `moved` blocks for new refactors.** Keep `state mv` for one-off ad-hoc cleanups.

We cover `moved` blocks in detail in notebook 07.


### `terraform state rm` — forgetting

Drop a resource from state *without destroying the cloud object*. The cloud resource keeps running; Terraform just stops managing it.

```bash
$ terraform state rm aws_instance.legacy
```

When to reach for this:

- Migrating a resource to a different Terraform configuration (drop here, import there).
- Adopting unmanaged infrastructure that Terraform created but should no longer track (rare).
- Cleaning up after a botched import — state has a wrong entry that doesn't match any HCL.

**What it doesn't do:** delete the cloud resource. The resource is still in AWS; running `terraform apply` after this will either create a duplicate (if the HCL still defines it) or leave it as an orphan (if the HCL is also removed).

A common combo for moving a resource between configurations: `terraform state rm` here, then `terraform import` (or `import` block) in the destination.


### `terraform import` — adopting existing resources

You created an S3 bucket through the AWS console six months ago. Now you want it managed by Terraform. `terraform import` brings an existing cloud resource into state under a specific address:

```hcl
# First, write the resource block in HCL — empty arguments are fine:
resource "aws_s3_bucket" "imported_logs" {
  bucket = "my-existing-logs-bucket"
}
```

```bash
$ terraform import aws_s3_bucket.imported_logs my-existing-logs-bucket
```

After this, state has the bucket's full attributes; the resource block in HCL is matched against state. The next `terraform plan` reports any *config drift* — attributes the cloud has that the HCL doesn't yet declare. Fill in the HCL to match (or accept the drift), and you're done.

**The pitfalls:**

- **Forgetting the resource block.** Import requires a target address to import *to*; the block has to exist in HCL first.
- **Resource IDs vary by type.** S3 wants the bucket name. EC2 wants the instance ID. The provider docs list the import syntax for every resource type.
- **No bulk import.** Importing 200 resources one-by-one is painful. Tools like `terraformer` (generate HCL + state from cloud) help but require careful review.

**The `import` block (Terraform 1.5+):** the modern, declarative form. We cover it in notebook 07. For one-off imports, the command is still fine.


## The bootstrap problem

Setting up an S3 backend requires an S3 bucket and a DynamoDB table. But those resources should themselves be managed by Terraform. Catch-22: the configuration that creates the backend can't store its state in the backend it's creating.

The standard solution is a **bootstrap configuration** with local state, applied once:

```hcl
# bootstrap/main.tf
terraform {
  required_version = ">= 1.6"
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
}

provider "aws" {
  region = "us-east-1"
}

resource "aws_s3_bucket" "tfstate" {
  bucket = "myorg-terraform-state"

  lifecycle {
    prevent_destroy = true
  }
}

resource "aws_s3_bucket_versioning" "tfstate" {
  bucket = aws_s3_bucket.tfstate.id
  versioning_configuration {
    status = "Enabled"
  }
}

resource "aws_s3_bucket_server_side_encryption_configuration" "tfstate" {
  bucket = aws_s3_bucket.tfstate.id

  rule {
    apply_server_side_encryption_by_default {
      sse_algorithm = "AES256"
    }
  }
}

resource "aws_dynamodb_table" "tflocks" {
  name         = "myorg-terraform-locks"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"

  attribute {
    name = "LockID"
    type = "S"
  }

  lifecycle {
    prevent_destroy = true
  }
}
```

Apply this once with local state. Commit the resulting `terraform.tfstate` to a **separate, private repository** (or import it into the same backend once it exists — chicken-and-egg, but possible).

From then on, every other Terraform configuration in the organization uses the S3 backend pointing at this bucket and table. The bootstrap config itself is rarely changed — versioning + prevent_destroy means its blast radius stays small.

A common variant: put the bootstrap *output* into Terraform Cloud or a separate state, and have the rest of the org's configurations read it via `terraform_remote_state`.


## Forward

Notebook four turns to **Variables, Locals & Expressions** — the input side of Terraform. Input variables with types and validation. Locals for derived values and naming. Outputs as the cross-module interface. The type constraints (primitives, collections, objects). Conditional expressions and the `for` expression for transforming collections. And the built-in function library for strings, files, encoding, and collections — the workhorse functions you reach for in every real configuration.
